<a href="https://colab.research.google.com/github/Arobnett/HDX-sources-and-more-API-connection/blob/main/CSCAN_3rd_party_sources.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install Excel support for reading/writing xlsx metadata previews.
!pip -q install openpyxl requests pandas

In [ ]:
# Import standard libraries for HTTP downloads, timestamps, hashing, and file handling.
import os
import re
import json
import hashlib
import requests
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path

In [ ]:
# Create a local Colab landing folder for raw public third-party source files.
RAW_DIR = Path("/content/hdx_raw_sources")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Define a browser-like user agent so public sites are less likely to reject the request.
HEADERS = {
    "User-Agent": "Mozilla/5.0 State_CA_OCS_MSU_public_source_downloader"
}

# Define the HDX CKAN API base URL.
HDX_API_BASE = "https://data.humdata.org/api/3/action"

In [ ]:
# Register the public third-party sources we want Colab to acquire.
sources = [
    {
        "source_id": "acled_political_violence_events_and_fatalities",
        "dataset_slug": "political-violence-events-and-fatalities",
        "expected_filename": "political-violence-events-and-fatalities.xlsx",
        "file_type": "xlsx",
        "source_family": "external_conflict_acled",
    },
    {
        "source_id": "acled_demonstration_events",
        "dataset_slug": "demonstration-events",
        "expected_filename": "demonstration-events.xlsx",
        "file_type": "xlsx",
        "source_family": "external_conflict_acled",
    },
    {
        "source_id": "acled_civilian_targeting_events_and_fatalities",
        "dataset_slug": "civilian-targeting-events-and-fatalities",
        "expected_filename": "civilian-targeting-events-and-fatalities.xlsx",
        "file_type": "xlsx",
        "source_family": "external_conflict_acled",
    },
    {
        "source_id": "worldriskindex_trend",
        "direct_url": "https://data.humdata.org/dataset/1efb6ee7-051a-440f-a2cf-e652fecccf73/resource/3a2320fa-41b4-4dda-a847-3f397d865378/download/worldriskindex-trend.csv",
        "expected_filename": "worldriskindex-trend.csv",
        "file_type": "csv",
        "source_family": "external_disaster_risk",
    },
    {
        "source_id": "worldriskindex_meta",
        "direct_url": "https://data.humdata.org/dataset/1efb6ee7-051a-440f-a2cf-e652fecccf73/resource/6aca2f09-ab01-4ff7-a17d-8cac9686cbba/download/worldriskindex-meta.xlsx",
        "expected_filename": "worldriskindex-meta.xlsx",
        "file_type": "xlsx",
        "source_family": "external_disaster_risk",
    },
    {
        "source_id": "inform_risk_index_trends",
        "direct_url": "https://data.humdata.org/dataset/f5ec2ee7-8a1b-49b4-864b-70bdb582a022/resource/b1d4a203-ef6e-44f7-9895-17c127aeaaee/download/inform_risk_index_trends.csv",
        "expected_filename": "inform_risk_index_trends.csv",
        "file_type": "csv",
        "source_family": "external_inform_risk",
    },
    {
        "source_id": "views_conflict_forecasts_country_month",
        "direct_url": "https://data.humdata.org/dataset/302a6645-ebc9-4d64-a673-e4730c4fc605/resource/df60f49d-5ead-48c1-aac5-e752f7278959/download/views-conflict-forecasts-country-month.csv",
        "expected_filename": "views-conflict-forecasts-country-month.csv",
        "file_type": "csv",
        "source_family": "external_conflict_forecast",
    },
    {
        "source_id": "gdacs_rss_information",
        "direct_url": "https://data.humdata.org/dataset/a87f96f8-16e6-4d51-872c-cfa54a8251ec/resource/4ef001d1-7888-4f5d-98ce-0ca8006787f7/download/gdacs_rss_information.csv",
        "expected_filename": "gdacs_rss_information.csv",
        "file_type": "csv",
        "source_family": "external_disaster_alert",
    },
    {
        "source_id": "who_covid_global_daily_data",
        "direct_url": "https://srhdpeuwpubsa.blob.core.windows.net/whdh/COVID/WHO-COVID-19-global-daily-data.csv",
        "expected_filename": "WHO-COVID-19-global-daily-data.csv",
        "file_type": "csv",
        "source_family": "external_health",
    },
    {
        "source_id": "who_world_health_statistics_2026",
        "direct_url": "https://xmart-api-public.who.int/DEX_CMS/WHS2026_DATADOWNLOAD?$format=csv&$filter=3+eq+3&$select=IndicatorName,IndicatorCode,Location,LocationCode,Year,Disaggregation,NumericValue,DisplayValue,Comments&$orderby=IndicatorName+asc,Location+asc,Year+asc,Disaggregation+asc",
        "expected_filename": "WHS2026_DATADOWNLOAD_as_of_2026_08_02.csv",
        "file_type": "csv",
        "source_family": "external_health",
    },
]

In [ ]:
# Find a downloadable resource URL from an HDX dataset slug when no direct URL is known.
def find_hdx_resource_url(dataset_slug, expected_filename=None):
    # Call HDX package_show to retrieve dataset metadata and resources.
    response = requests.get(
        f"{HDX_API_BASE}/package_show",
        params={"id": dataset_slug},
        headers=HEADERS,
        timeout=90,
    )

    # Raise an error if HDX did not return a successful API response.
    response.raise_for_status()

    # Extract the dataset object from the CKAN response.
    dataset = response.json()["result"]

    # Pull the resources list from the dataset metadata.
    resources = dataset.get("resources", [])

    # Prefer the resource whose URL or name matches the expected filename.
    if expected_filename:
        for resource in resources:
            name = str(resource.get("name", "")).lower()
            url = str(resource.get("url", "")).lower()
            if expected_filename.lower() in name or expected_filename.lower() in url:
                return resource.get("url"), resource

    # Otherwise use the first XLSX/CSV resource available.
    for resource in resources:
        fmt = str(resource.get("format", "")).lower()
        url = str(resource.get("url", "")).lower()
        if fmt in ["xlsx", "csv"] or url.endswith((".xlsx", ".csv")):
            return resource.get("url"), resource

    # Fail clearly if no usable resource was found.
    raise ValueError(f"No CSV/XLSX resource found for dataset_slug={dataset_slug}")

In [ ]:
# Download one source and return Bronze-style metadata.
def download_source(source):
    # Resolve the final download URL from either a direct URL or HDX dataset metadata.
    if source.get("direct_url"):
        download_url = source["direct_url"]
        resource_metadata = {}
    else:
        download_url, resource_metadata = find_hdx_resource_url(
            source["dataset_slug"],
            source.get("expected_filename"),
        )

    # Build a stable local output path using the expected filename.
    output_path = RAW_DIR / source["expected_filename"]

    # Stream the file so large XLSX/CSV downloads do not need to sit fully in memory.
    with requests.get(download_url, headers=HEADERS, stream=True, timeout=180) as response:
        response.raise_for_status()
        with open(output_path, "wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)

    # Read the raw bytes for hash and byte count metadata.
    file_bytes = output_path.read_bytes()

    # Return a metadata row compatible with your Bronze source artifact concept.
    return {
        "source_id": source["source_id"],
        "source_family": source["source_family"],
        "filename": output_path.name,
        "local_path": str(output_path),
        "file_type": source["file_type"],
        "acquisition_method": "colab_public_download",
        "canonical_source_url": download_url,
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        "byte_count": len(file_bytes),
        "content_sha256": hashlib.sha256(file_bytes).hexdigest(),
        "resource_id": resource_metadata.get("id"),
        "resource_name": resource_metadata.get("name"),
    }

In [ ]:
# Download every registered public source and collect metadata.
metadata_rows = []

# Loop through each source one-by-one so failures are easy to identify.
for source in sources:
    try:
        print(f"Downloading: {source['source_id']}")
        metadata_rows.append(download_source(source))
        print("  success")
    except Exception as error:
        print(f"  failed: {error}")

# Convert the metadata list into a DataFrame for review.
metadata_df = pd.DataFrame(metadata_rows)

# Display the downloaded-source inventory.
metadata_df

Downloading: acled_political_violence_events_and_fatalities
  success
Downloading: acled_demonstration_events
  success
Downloading: acled_civilian_targeting_events_and_fatalities
  success
Downloading: worldriskindex_trend
  success
Downloading: worldriskindex_meta
  success
Downloading: inform_risk_index_trends
  success
Downloading: views_conflict_forecasts_country_month
  success
Downloading: gdacs_rss_information
  success
Downloading: who_covid_global_daily_data
  success
Downloading: who_world_health_statistics_2026
  success


,source_id,source_family,filename,local_path,file_type,acquisition_method,canonical_source_url,retrieved_at_utc,byte_count,content_sha256,resource_id,resource_name
0,acled_political_violence_events_and_fatalities,external_conflict_acled,political-violence-events-and-fatalities.xlsx,/content/hdx_raw_sources/political-violence-ev...,xlsx,colab_public_download,https://data.humdata.org/dataset/3e6bfc98-f837...,2026-08-05T14:28:07.307430+00:00,43873684,204d90db6085c20ca9293ece5cea0735ad0bda3c2ba31e...,99a32d01-d0ca-4f57-a0f5-cb6b5f01f14f,political-violence-events-and-fatalities_as-of...
1,acled_demonstration_events,external_conflict_acled,demonstration-events.xlsx,/content/hdx_raw_sources/demonstration-events....,xlsx,colab_public_download,https://data.humdata.org/dataset/c946742e-cf48...,2026-08-05T14:28:09.062414+00:00,40073182,6351c733aa7ed34cb24431a6fcb4432cd08ca6b0341ef5...,4f98e767-d28d-4711-b575-ded17a0421ce,demonstration-events_as-of-2026-07-24.xlsx
2,acled_civilian_targeting_events_and_fatalities,external_conflict_acled,civilian-targeting-events-and-fatalities.xlsx,/content/hdx_raw_sources/civilian-targeting-ev...,xlsx,colab_public_download,https://data.humdata.org/dataset/921fe117-7158...,2026-08-05T14:28:10.660973+00:00,43646545,f0d001e25197ed3033e2e3955a3014cfb62f6073d55d55...,fb139931-641f-4428-be77-429972878e19,civilian-targeting-events-and-fatalities_as-of...
3,worldriskindex_trend,external_disaster_risk,worldriskindex-trend.csv,/content/hdx_raw_sources/worldriskindex-trend.csv,csv,colab_public_download,https://data.humdata.org/dataset/1efb6ee7-051a...,2026-08-05T14:28:11.927518+00:00,7044964,098bdc14a76fcd06eb17c4b0c4bc9988301573e5fbd0e1...,None,None
4,worldriskindex_meta,external_disaster_risk,worldriskindex-meta.xlsx,/content/hdx_raw_sources/worldriskindex-meta.xlsx,xlsx,colab_public_download,https://data.humdata.org/dataset/1efb6ee7-051a...,2026-08-05T14:28:12.806396+00:00,41138,aab3a0e95a413c727f21fb5439fe95464e5ba020d33280...,None,None
5,inform_risk_index_trends,external_inform_risk,inform_risk_index_trends.csv,/content/hdx_raw_sources/inform_risk_index_tre...,csv,colab_public_download,https://data.humdata.org/dataset/f5ec2ee7-8a1b...,2026-08-05T14:28:13.836444+00:00,397862,26b25ead3eccee2d583ad158b2023bba2745ea71be26b8...,None,None
6,views_conflict_forecasts_country_month,external_conflict_forecast,views-conflict-forecasts-country-month.csv,/content/hdx_raw_sources/views-conflict-foreca...,csv,colab_public_download,https://data.humdata.org/dataset/302a6645-ebc9...,2026-08-05T14:28:14.487540+00:00,51415,f66c5384836a16a79f024802ad857516ff0459d8377817...,None,None
7,gdacs_rss_information,external_disaster_alert,gdacs_rss_information.csv,/content/hdx_raw_sources/gdacs_rss_information...,csv,colab_public_download,https://data.humdata.org/dataset/a87f96f8-16e6...,2026-08-05T14:28:15.281660+00:00,129028,258ef2216174e49f474450c5123e05696dbefd543f76e3...,None,None
8,who_covid_global_daily_data,external_health,WHO-COVID-19-global-daily-data.csv,/content/hdx_raw_sources/WHO-COVID-19-global-d...,csv,colab_public_download,https://srhdpeuwpubsa.blob.core.windows.net/wh...,2026-08-05T14:28:21.218994+00:00,26097232,6063fda90dba3439f551c0bfadce53bb437d9982056cd1...,None,None
9,who_world_health_statistics_2026,external_health,WHS2026_DATADOWNLOAD_as_of_2026_08_02.csv,/content/hdx_raw_sources/WHS2026_DATADOWNLOAD_...,csv,colab_public_download,https://xmart-api-public.who.int/DEX_CMS/WHS20...,2026-08-05T14:28:22.604466+00:00,1444612,6639651b53b7059dcdcbb47fce87db469df15ed899182e...,None,None


In [ ]:
# Save the metadata inventory beside the raw downloaded files.
metadata_path = RAW_DIR / "hdx_colab_download_manifest.csv"

# Write the manifest as CSV for upload into Databricks Bronze metadata tracking.
metadata_df.to_csv(metadata_path, index=False)

# Show the manifest path.
metadata_path

PosixPath('/content/hdx_raw_sources/hdx_colab_download_manifest.csv')

In [ ]:
# Inspect Excel workbook sheet names so multi-tab HDX files are visible before Databricks landing.
for row in metadata_rows:
    if row["file_type"] == "xlsx":
        try:
            excel_file = pd.ExcelFile(row["local_path"])
            print(row["filename"], "=>", excel_file.sheet_names)
        except Exception as error:
            print(row["filename"], "=> could not inspect:", error)

political-violence-events-and-fatalities.xlsx => ['TOU', 'Non_HRP', 'HRP_1', 'HRP_2']
demonstration-events.xlsx => ['TOU', 'Non_HRP', 'HRP_1', 'HRP_2']
civilian-targeting-events-and-fatalities.xlsx => ['TOU', 'Non_HRP', 'HRP_1', 'HRP_2']
worldriskindex-meta.xlsx => ['WorldRiskIndex', 'Imputation Variables']


In [ ]:
# Zip all downloaded raw files and the manifest for easy manual upload to SharePoint or Databricks.
!cd /content && zip -r hdx_raw_sources.zip hdx_raw_sources

  adding: hdx_raw_sources/ (stored 0%)
  adding: hdx_raw_sources/political-violence-events-and-fatalities.xlsx (deflated 12%)
  adding: hdx_raw_sources/WHO-COVID-19-global-daily-data.csv (deflated 86%)
  adding: hdx_raw_sources/WHS2026_DATADOWNLOAD_as_of_2026_08_02.csv (deflated 90%)
  adding: hdx_raw_sources/views-conflict-forecasts-country-month.csv (deflated 74%)
  adding: hdx_raw_sources/inform_risk_index_trends.csv (deflated 91%)
  adding: hdx_raw_sources/hdx_colab_download_manifest.csv (deflated 63%)
  adding: hdx_raw_sources/civilian-targeting-events-and-fatalities.xlsx (deflated 12%)
  adding: hdx_raw_sources/demonstration-events.xlsx (deflated 10%)
  adding: hdx_raw_sources/gdacs_rss_information.csv (deflated 85%)
  adding: hdx_raw_sources/worldriskindex-meta.xlsx (deflated 8%)
  adding: hdx_raw_sources/worldriskindex-trend.csv (deflated 62%)


In [ ]:
!python transform_public_sources_to_country_month_year.py \
  --input-dir /content/hdx_raw_sources \
  --output-dir /content/country_month_year_outputs

python3: can't open file '/content/transform_public_sources_to_country_month_year.py': [Errno 2] No such file or directory
